# ⚡ BoneRAG — Free Google Colab GPU Backend (pyngrok)

### 📌 Hướng dẫn:
1. **Bật GPU**: `Runtime` → `Change runtime type` → `T4 GPU`
2. Điền **ngrok authtoken** vào ô `NGROK_TOKEN` ở Step 3 (lấy miễn phí tại https://dashboard.ngrok.com/authtokens)
3. Bấm **Runtime → Run all (Ctrl+F9)**
4. Copy link `https://xxxx.ngrok-free.app` → dán vào Vercel `VITE_API_BASE_URL` → Redeploy

In [ ]:
# [Step 1] Kiểm tra GPU & Cài đặt thư viện
!nvidia-smi
!pip install -q torch torchvision transformers open_clip_torch faiss-cpu pillow numpy huggingface_hub pyngrok

In [ ]:
# [Step 2] Tải mã nguồn BoneRAG mới nhất từ GitHub
import os
if not os.path.exists('/content/boneRAG'):
    !git clone https://github.com/thanhnghi-do-2k3/boneRAG.git /content/boneRAG
else:
    !cd /content/boneRAG && git pull
%cd /content/boneRAG
print('✅ Code đã sẵn sàng!')

In [ ]:
# [Step 3] ⚙️ ĐIỀN NGROK TOKEN VÀO ĐÂY (lấy miễn phí tại https://dashboard.ngrok.com/authtokens)
NGROK_TOKEN = "REDACTED"  # <-- Dán token ngrok của bạn vào đây

import subprocess, time, urllib.request
from pyngrok import ngrok, conf

PORT = 8088

# Cài ngrok token
if not NGROK_TOKEN:
    raise ValueError("⚠️ Chưa điền NGROK_TOKEN! Lấy miễn phí tại: https://dashboard.ngrok.com/authtokens")
conf.get_default().auth_token = NGROK_TOKEN

# ✅ Ghi log server ra file để tránh PIPE deadlock
log_file = open('/content/bonerag_server.log', 'w')
server_proc = subprocess.Popen(
    ['python3', 'demo-app/server.py', '--host', '0.0.0.0', '--port', str(PORT)],
    stdout=log_file,
    stderr=log_file,
)

print(f'⏳ Đang chờ BoneRAG server khởi động (PID={server_proc.pid})...')
print('   (Log server: /content/bonerag_server.log)')
for i in range(40):
    time.sleep(3)
    if server_proc.poll() is not None:
        print(f'\n❌ Server bị crash! Nội dung log:')
        with open('/content/bonerag_server.log') as f:
            print(f.read()[-3000:])
        raise RuntimeError('Server crashed.')
    try:
        urllib.request.urlopen(f'http://127.0.0.1:{PORT}/api/records', timeout=2)
        print(f'\n✅ Server đã sẵn sàng! (sau {(i+1)*3}s)')
        break
    except Exception:
        print(f'   [{i+1}/40] Đang chờ server...', end='\r')
else:
    print('\n⚠️ Timeout! Log server:')
    with open('/content/bonerag_server.log') as f:
        print(f.read()[-3000:])
    raise RuntimeError('Server timeout.')

In [ ]:
# [Step 4] Mở ngrok tunnel & in ra Public URL
from pyngrok import ngrok

# Đóng tunnel cũ nếu có
ngrok.kill()

tunnel = ngrok.connect(PORT, bind_tls=True)
PUBLIC_URL = tunnel.public_url

print()
print('=' * 65)
print('🚀 BACKEND GPU ĐANG CHẠY — COPY LINK NÀY VÀO VERCEL:')
print('=' * 65)
print(f'  VITE_API_BASE_URL = {PUBLIC_URL}')
print('=' * 65)
print('📌 Sau khi dán vào Vercel → Save → Redeploy!')
print('⚠️  Giữ ô này đang chạy. Đừng đóng Colab!')

# Giữ ngrok tunnel sống
import signal, time as _t
try:
    while True:
        _t.sleep(30)
        # Ping server để giữ Colab không idle
        urllib.request.urlopen(f'http://127.0.0.1:{PORT}/api/records', timeout=5)
except KeyboardInterrupt:
    print('\n🛑 Đã dừng tunnel.')
    ngrok.kill()